# Evaluate score separation, event recall and alarm burden

Run the pipeline and dashboard export first. See [methodology](../docs/methodology.md) for protocol v2.

In [1]:
from pathlib import Path
import sys, json
import pandas as pd
import plotly.express as px
import plotly.io as pio
pio.renderers.default = 'plotly_mimetype'
ROOT = Path.cwd() if (Path.cwd()/'results').exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT/'src'))
experiment = json.loads((ROOT/'results/experiment.json').read_text())
assert experiment['protocol_version'] == 2


## Compare every frozen method on the chronological benchmark

In [2]:
comparison = pd.read_csv(ROOT/'results/model_comparison.csv')
comparison[['method','precision','recall','f1','pr_auc_ap','leaks_detected','false_alarms','mean_delay_hours']]

,method,precision,recall,f1,pr_auc_ap,leaks_detected,false_alarms,mean_delay_hours
0,Robust baseline,0.163507,0.075547,0.103345,0.165456,15,158,33.122222
1,IQR baseline,0.165524,0.198175,0.180384,0.164127,19,330,27.307018
2,Isolation Forest,0.167068,0.202737,0.183182,0.170162,19,397,22.346491
3,PCA reconstruction,0.297480,0.120620,0.171644,0.216230,16,135,36.359375
4,Local Outlier Factor,0.233266,0.125912,0.163546,0.196875,17,207,28.617647


## Does the score retain precision as recall grows?

In [3]:
curves = pd.read_csv(ROOT/'results/precision_recall_curves.csv')
px.line(curves, x='recall', y='precision', color='method', range_x=[0,1], range_y=[0,1])

## Could frequent alarms explain the apparent event recall?

In [4]:
comparison[['method','event_recall','shift_control_mean_recall','shift_control_p05_recall','shift_control_p95_recall','false_alarms_per_week']]

,method,event_recall,shift_control_mean_recall,shift_control_p05_recall,shift_control_p95_recall,false_alarms_per_week
0,Robust baseline,0.789474,0.801579,0.684211,0.894737,3.030137
1,IQR baseline,1.000000,0.961316,0.894737,1.000000,6.328767
2,Isolation Forest,1.000000,0.960789,0.894737,1.000000,7.613699
3,PCA reconstruction,0.842105,0.677105,0.526316,0.842105,2.589041
4,Local Outlier Factor,0.894737,0.789737,0.684211,0.894737,3.969863


## Inspect each PCA event, including misses and contributing sensors

In [5]:
detections = pd.read_csv(ROOT/'results/event_detections.csv')
detections[detections.method.eq('PCA reconstruction')][['pipe','start','detected_at','delay_hours','contributing_sensors','signed_residual_z']]

,pipe,start,detected_at,delay_hours,contributing_sensors,signed_residual_z
57,p523,2019-01-15 23:00:00,2019-01-16 15:45:00,16.750000,"pressure:n506, pressure:n105, pressure:n469, p...","pressure:n506=-6.15, pressure:n105=-4.81, pres..."
58,p827,2019-01-24 18:30:00,2019-01-24 19:30:00,1.000000,"pressure:n726, pressure:n722, pressure:n752, p...","pressure:n726=-8.47, pressure:n722=-7.18, pres..."
59,p280,2019-02-10 13:05:00,2019-02-10 18:45:00,5.666667,"pressure:n726, pressure:n752, pressure:n722, p...","pressure:n726=+7.68, pressure:n752=+6.56, pres..."
60,p653,2019-03-03 13:10:00,2019-03-05 14:00:00,48.833333,"level:T1, pressure:n31, flow:PUMP_1, pressure:n4","level:T1=+8.17, pressure:n31=+5.69, flow:PUMP_..."
61,p710,2019-03-24 14:15:00,2019-03-27 07:15:00,65.000000,"flow:PUMP_1, level:T1, pressure:n31, pressure:n4","flow:PUMP_1=+7.59, level:T1=-4.01, pressure:n3..."
62,p514,2019-04-02 20:40:00,2019-04-04 13:30:00,40.833333,"flow:PUMP_1, level:T1, pressure:n31, pressure:n4","flow:PUMP_1=-2217.89, level:T1=+13.75, pressur..."
63,p331,2019-04-20 10:10:00,2019-04-23 09:15:00,71.083333,"pressure:n4, flow:PUMP_1, pressure:n1, level:T1","pressure:n4=-8.42, flow:PUMP_1=+7.93, pressure..."
64,p193,2019-05-19 10:40:00,2019-05-22 07:00:00,68.333333,"flow:PUMP_1, pressure:n1, level:T1, pressure:n410","flow:PUMP_1=+13.32, pressure:n1=-7.78, level:T..."
65,p277,2019-05-30 21:55:00,2019-05-31 17:15:00,19.333333,"flow:PUMP_1, level:T1, pressure:n54, pressure:...","flow:PUMP_1=-2218.23, level:T1=+10.23, pressur..."
66,p142,2019-06-12 19:55:00,2019-06-14 07:30:00,35.583333,"flow:PUMP_1, pressure:n54, pressure:n429, pres...","flow:PUMP_1=-10.29, pressure:n54=-8.72, pressu..."


## How much does the detection window change conclusions?

In [6]:
sensitivity = pd.read_csv(ROOT/'results/horizon_sensitivity.csv')
sensitivity[['method','horizon_hours','leaks_detected','false_alarms','mean_delay_hours']]

,method,horizon_hours,leaks_detected,false_alarms,mean_delay_hours
0,Robust baseline,24,6,167,7.736111
1,Robust baseline,72,15,158,33.122222
2,Robust baseline,168,17,156,42.578431
3,IQR baseline,24,10,339,14.875000
4,IQR baseline,72,19,330,27.307018
5,IQR baseline,168,19,330,27.307018
6,Isolation Forest,24,13,403,12.602564
7,Isolation Forest,72,19,397,22.346491
8,Isolation Forest,168,19,397,22.346491
9,PCA reconstruction,24,6,145,11.347222


## Inspect validation evidence rather than tune against the test year

In [7]:
validation = pd.read_csv(ROOT/'results/validation_calibration.csv')
validation[['method','quantile','event_f1','leaks_detected','false_alarms']]

,method,quantile,event_f1,leaks_detected,false_alarms
0,Robust baseline,0.950,0.088889,2,41
1,Robust baseline,0.975,0.066667,1,27
2,Robust baseline,0.990,0.000000,0,8
3,Robust baseline,0.995,0.000000,0,6
4,Robust baseline,0.999,0.000000,0,3
5,IQR baseline,0.950,0.052632,1,35
6,IQR baseline,0.975,0.000000,0,11
7,IQR baseline,0.990,0.000000,0,4
8,IQR baseline,0.995,0.000000,0,2
9,IQR baseline,0.999,0.000000,0,1


## Interpretation and limits

This is protocol v2 after a correctness review, with no 2019-based threshold search. The benchmark has already been inspected, so it is not an untouched external test. There are only two validation onsets. Mean delay excludes misses; the timing-shift control is descriptive, not a significance test. These detectors are not ready for operational deployment.